# Approximated Topology Visualizer

This notebook visualises the three approximated topology variants side-by-side with their originals.

## Node colour key (consistent throughout)
| Colour | Node type |
|--------|-----------|
| 🟢 Green `#4CAF50` | NPU (`h*`) |
| 🟠 Orange `#FF9800` | NVSwitch / NPU-switch (`v*`, `p*`) |
| 🩷 Pink `#E91E63` | NIC switch (`n*`) |
| 🔵 Blue `#2196F3` | ToR switch (`t*`) — **unchanged** |
| 🟣 Purple `#9C27B0` | Merged aggregator (`g*`) — **approximated** |
| 🔴 Red `#F44336` | Core switch (`c*`) — **unchanged** in `group_agg` |
| 🟥 Dark-red `#B71C1C` | Root switch (`r0`) — **approximated** in `group_agg_core` |

## Two approximation variants compared
- **`group_agg`**: collapse agg layer only → keep original core switches
- **`group_agg_core`**: collapse agg **and** core → single `r0` root


## 1 · Setup and Imports

In [ ]:
import sys, os, importlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

# ── path setup ──────────────────────────────────────────────────────────────
_APROX = os.path.abspath('.')                                    # .../aprox/
_NEW   = os.path.join(_APROX, '..', 'new')
for p in [_APROX, _NEW]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── original topology classes ────────────────────────────────────────────────
import create_topology; importlib.reload(create_topology)
from create_topology import FoldedClos, CustomizedDragonfly, Jellyfish

import utils; importlib.reload(utils)
from utils import visualize_topology, build_topology

# ── approximation modules ────────────────────────────────────────────────────
import aprox_topology_group_agg     as ga;   importlib.reload(ga)
import aprox_topology_group_agg_core as gac; importlib.reload(gac)

# ── visualisation helper ─────────────────────────────────────────────────────
import visualize_approx; importlib.reload(visualize_approx)
from visualize_approx import (
    visualize_approx_topology,
    visualize_approx_single_server,
    _classify, _COLORS, _LEGEND_LABELS,
)

print("Imports OK")


## 2 · FoldedClos — `group_agg` approximation (core kept)

Agg layer collapsed into one `g{pod}` per pod; original core switches (`c*`) preserved.

In [ ]:
fc_bw  = {'host_edge': 100, 'edge_agg': 100, 'agg_core': 100, 'intra_node': 900}
fc_lat = {'host_edge': 0.001, 'edge_agg': 0.01, 'agg_core': 0.05, 'intra_node': 0.0001}

fc = FoldedClos(K=6, bandwidth_config=fc_bw, latency_config=fc_lat,
                npus_per_node=1, intra_node_topology='switch', num_nvswitches=1)
fc.DesignFullTopology()
links_out = fc.LinksToG2ConfFile()
if isinstance(links_out, tuple):
    links_out = links_out[0]

ga_fc_links, ga_fc_bw, ga_fc_lat, ga_fc_meta = ga.approximate_foldedclos(fc)

fig, ax, G = visualize_approx_topology(
    fc, ga_fc_links, ga_fc_bw, ga_fc_meta,
    title="FoldedClos K=4  [group_agg]  — agg collapsed, core unchanged",
    figsize=(16, 10), show_bw=True)
plt.show()

print(f"ToR switches   : {len(ga_fc_meta['tor_switches'])}")
print(f"Merged agg (g*): {sorted(ga_fc_meta['upper_switches'])}")
print(f"Core switches  : {len(ga_fc_meta.get('core_switches', []))} (unchanged)")
print(f"tor→group sample: {dict(list(ga_fc_meta['tor_to_group'].items())[:4])}")


## 3 · FoldedClos — `group_agg_core` approximation (agg + core both collapsed)

Both agg **and** core replaced: `g{pod}` per pod + single `r0` root.

In [ ]:
# reuse the same fc object built in cell 2
gac_fc_links, gac_fc_bw, gac_fc_lat, gac_fc_meta = gac.approximate_foldedclos(fc)

fig, ax, G = visualize_approx_topology(
    fc, gac_fc_links, gac_fc_bw, gac_fc_meta,
    title="FoldedClos K=4  [group_agg_core]  — agg + core collapsed into g* + r0",
    figsize=(16, 10), show_bw=True)
plt.show()

print(f"ToR switches    : {len(gac_fc_meta['tor_switches'])}")
print(f"Upper-fabric    : {sorted(gac_fc_meta['upper_switches'])}")
print(f"group→core map  : {gac_fc_meta['group_to_core']}")


## 4 · FoldedClos — 3-way comparison: Original vs group_agg vs group_agg_core

In [ ]:
import networkx as nx

def _count_nodes_links(links):
    nodes = set()
    for s, d in links.values():
        nodes.add(s); nodes.add(d)
    switches = [n for n in nodes if not n.startswith('h')]
    return len(switches), len(links) // 2   # links/2 = undirected count

orig_sw,  orig_l  = _count_nodes_links(fc.links)
ga_sw,   ga_l    = _count_nodes_links(ga_fc_links)
gac_sw,  gac_l   = _count_nodes_links(gac_fc_links)

print("=" * 62)
print(f"{'Metric':<30} {'Original':>10} {'group_agg':>10} {'g+core':>10}")
print("-" * 62)
print(f"{'Switch nodes':<30} {orig_sw:>10} {ga_sw:>10} {gac_sw:>10}")
print(f"{'Undirected links':<30} {orig_l:>10} {ga_l:>10} {gac_l:>10}")
print(f"{'Upper-fabric depth':<30} {'3 (e+a+c)':>10} {'2 (e+g+c)':>10} {'2 (e+g+r0)':>10}")
print(f"{'ECMP needed?':<30} {'yes':>10} {'no':>10} {'no':>10}")
print("=" * 62)

# Single-server view for group_agg
fig, ax = visualize_approx_single_server(
    fc, ga_fc_links, ga_fc_bw, ga_fc_meta,
    server_idx=1, figsize=(12, 9))
ax.set_title("FoldedClos group_agg — Server 1 stack (NPU → NVSwitch → ToR → g* → core)",
             fontsize=12, fontweight='bold')
plt.show()


## 5 · Dragonfly — approximation (identical for both variants)

In [ ]:
df_bw  = {'host_switch': 100, 'intra_group': 200, 'inter_group': 100, 'intra_node': 900}
df_lat = {'host_switch': 0.001, 'intra_group': 0.01, 'inter_group': 0.1, 'intra_node': 0.0001}

df = CustomizedDragonfly(G=4, A=4, h=2, concentration=1,
                         bandwidth_config=df_bw, latency_config=df_lat,
                         npus_per_node=2, intra_node_topology='switch', num_nvswitches=1)
df.DesignFullTopology()
df.LinksToG2ConfFile()

ga_df_links, ga_df_bw, ga_df_lat, ga_df_meta = ga.approximate_dragonfly(df)

fig, ax, G = visualize_approx_topology(
    df, ga_df_links, ga_df_bw, ga_df_meta,
    title="Dragonfly G=4 A=4 h=2  —  intra-group mesh → g{i},  inter-group → r0",
    figsize=(16, 10), show_bw=True)
plt.show()

print(f"ToR switches    : {sorted(ga_df_meta['tor_switches'])}")
print(f"Group agg (g*)  : {[n for n in ga_df_meta['upper_switches'] if n.startswith('g')]}")
print(f"Root            : {[n for n in ga_df_meta['upper_switches'] if n == 'r0']}")
print(f"tor→group sample: {dict(list(ga_df_meta['tor_to_group'].items())[:4])}")
print(f"group→root      : {ga_df_meta['group_to_core']}")


In [ ]:
# Single-server view for Dragonfly
fig, ax = visualize_approx_single_server(
    df, ga_df_links, ga_df_bw, ga_df_meta,
    server_idx=1, figsize=(12, 9))
ax.set_title("Dragonfly — Server 1 stack  (NPU → NVSwitch → ToR → g* → r0)",
             fontsize=12, fontweight='bold')
plt.show()


## 6 · Jellyfish — star approximation

No natural grouping → single `r0` above all ToRs. BW on each `ToR→r0` link = sum of all switch-switch links on that ToR.

In [ ]:
jf_bw  = {'host_switch': 100, 'switch_switch': 200, 'intra_node': 900}
jf_lat = {'host_switch': 0.001, 'switch_switch': 0.01, 'intra_node': 0.0001}

jf = Jellyfish(num_switches=8, degree=3, num_hosts_per_switch=1,
               bandwidth_config=jf_bw, latency_config=jf_lat,
               npus_per_node=2, intra_node_topology='switch', num_nvswitches=1)
jf.DesignFullTopology()
jf.LinksToG2ConfFile()

ga_jf_links, ga_jf_bw, ga_jf_lat, ga_jf_meta = ga.approximate_jellyfish(jf)

fig, ax, G = visualize_approx_topology(
    jf, ga_jf_links, ga_jf_bw, ga_jf_meta,
    title="Jellyfish 8sw deg=3  —  random mesh → r0 star",
    figsize=(12, 8), show_bw=True)
plt.show()

# Aggregated BW per ToR → r0
r0_bw = {s: bw for lid, (s, d) in ga_jf_links.items()
          if d == 'r0' for bw in [ga_jf_bw[lid]]}
print("Per-ToR uplink BW to r0:")
for tor, bw in sorted(r0_bw.items()):
    print(f"  {tor} → r0 : {bw:.0f} GB/s")


## 7 · Link Bandwidth Preservation Analysis

Verify that the approximation preserves total available bandwidth per node.

In [ ]:
def _sum_bw(bw_dict):
    return sum(bw_dict.values())

def _tor_uplink_bw(links, bw, tor_set):
    """Sum BW of all links from a ToR to a non-ToR, non-NPU node."""
    result = {}
    for lid, (s, d) in links.items():
        if s in tor_set and not d.startswith('h') and d not in tor_set:
            result[s] = result.get(s, 0) + bw.get(lid, 0)
    return result

cases = [
    ("FC original",    fc.links,         fc.link_bandwidths,    set(fc.edge_switches)),
    ("FC group_agg",   ga_fc_links,      ga_fc_bw,             ga_fc_meta['tor_switches']),
    ("FC g+core",      gac_fc_links,     gac_fc_bw,            gac_fc_meta['tor_switches']),
    ("DF original",    df.links,         df.link_bandwidths,    set(f't{i}' for i in range(1, df.total_num_switches+1))),
    ("DF group_agg",   ga_df_links,      ga_df_bw,             ga_df_meta['tor_switches']),
    ("JF original",    jf.links,         jf.link_bandwidths,    set(f't{i}' for i in range(1, jf.num_switches+1))),
    ("JF group_agg",   ga_jf_links,      ga_jf_bw,             ga_jf_meta['tor_switches']),
]

labels       = [c[0] for c in cases]
total_bw     = [_sum_bw(c[2]) for c in cases]
tor_uplink   = [sum(_tor_uplink_bw(c[1], c[2], c[3]).values()) for c in cases]

x = range(len(labels))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.bar(x, total_bw, color='#2196F3', alpha=0.8)
ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax1.set_ylabel("Total link BW (sum, both directions)"); ax1.set_title("Total BW in topology")

ax2.bar(x, tor_uplink, color='#9C27B0', alpha=0.8)
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax2.set_ylabel("Total ToR uplink BW (all ToRs combined)"); ax2.set_title("ToR → upper-fabric BW")

plt.tight_layout(); plt.show()

print(f"\n{'Case':<18} {'Total BW':>12} {'ToR uplink':>12}")
print("-" * 44)
for lbl, tbw, tup in zip(labels, total_bw, tor_uplink):
    print(f"{lbl:<18} {tbw:>12.0f} {tup:>12.0f}")


## 8 · Path Length Distribution Comparison

For FoldedClos K=4 with `npus_per_node=1` (small, so all-pairs is fast), compare hop counts between original ECMP paths and the approximated paths.

In [ ]:
# Build a small FC (npus_per_node=1) for fast all-pairs path enumeration
fc_small = FoldedClos(K=4, bandwidth_config=fc_bw, latency_config=fc_lat,
                      npus_per_node=1, intra_node_topology='switch', num_nvswitches=1)
fc_small.DesignFullTopology()
lout = fc_small.LinksToG2ConfFile()
if isinstance(lout, tuple):
    lout = lout[0]

ga_s_links,  ga_s_bw,  ga_s_lat,  ga_s_meta  = ga.approximate_foldedclos(fc_small)
gac_s_links, gac_s_bw, gac_s_lat, gac_s_meta = gac.approximate_foldedclos(fc_small)

# ── All-pairs hop-count helper ──────────────────────────────────────────────
def _all_pairs_hop_lengths(links):
    G = nx.DiGraph()
    for s, d in links.values():
        G.add_edge(s, d)
    npus = sorted(n for n in G.nodes() if n.startswith('h'))
    sp   = dict(nx.all_pairs_shortest_path_length(G))
    return [
        sp[src][dst]
        for src in npus
        for dst in npus
        if src != dst and dst in sp.get(src, {})
    ]

orig_lens = _all_pairs_hop_lengths(fc_small.links)
ga_lens   = _all_pairs_hop_lengths(ga_s_links)
gac_lens  = _all_pairs_hop_lengths(gac_s_links)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
configs = [
    (orig_lens, "Original",                  '#2196F3'),
    (ga_lens,   "group_agg  (core kept)",    '#9C27B0'),
    (gac_lens,  "group_agg_core  (r0)",      '#B71C1C'),
]
for ax, (lens, title, color) in zip(axes, configs):
    bins = range(min(lens), max(lens) + 2)
    ax.hist(lens, bins=bins, color=color, alpha=0.82, edgecolor='white', linewidth=0.6)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel("Path length (hops)", fontsize=10)
    ax.set_ylabel("NPU pairs", fontsize=10)
    stats = (f"mean={np.mean(lens):.2f}  "
             f"min={min(lens)}  max={max(lens)}  "
             f"σ={np.std(lens):.2f}")
    ax.text(0.5, 0.96, stats, transform=ax.transAxes,
            ha='center', va='top', fontsize=8, color='#333333')

plt.suptitle("FoldedClos K=4 — All-pairs NPU path-length distribution",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'Variant':<28} {'mean':>7}  {'min':>4}  {'max':>4}  {'σ':>7}")
print("─" * 55)
for name, lens in [("Original", orig_lens),
                   ("group_agg (core kept)", ga_lens),
                   ("group_agg_core (r0)", gac_lens)]:
    print(f"{name:<28} {np.mean(lens):>7.2f}  "
          f"{min(lens):>4}  {max(lens):>4}  {np.std(lens):>7.2f}")